In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install torchinfo

# **Import Libraries**

In [3]:
import os
import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, 
                             roc_curve, auc, roc_auc_score, precision_recall_fscore_support)
from sklearn.preprocessing import label_binarize
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

# **Configuration**

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

data_dir = "/kaggle/input/multi-cancer/Multi Cancer/Multi Cancer/Cervical Cancer"
output_dir = "/kaggle/working/processed-dataset" 
checkpoints_path = "/kaggle/working/"
os.makedirs(checkpoints_path, exist_ok=True)

BATCH_SIZE = 32
LEARNING_RATE = 0.0001
EPOCHS = 100 
PATIENCE = 10
NUM_CLASSES = 5

Using Device: cuda


# **Data Loading and Preprocessing**

In [5]:
def load_data(root_dir):
    file_paths = []
    labels = []
    
    if not os.path.exists(root_dir):
        print(f"ERROR: Directory not found: {root_dir}")
        return pd.DataFrame() 

    classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    print(f"Classes found: {classes}")

    for class_name in classes:
        class_dir = os.path.join(root_dir, class_name)
        for root, _, files in os.walk(class_dir):
            for file in files:
                if file.lower().endswith(('.bmp', '.jpg', '.png', '.jpeg')):
                    file_paths.append(os.path.join(root, file))
                    labels.append(class_name)

    df = pd.DataFrame({"file_path": file_paths, "label": labels})
    return df

df = load_data(data_dir)

if df.empty:
    print("WARNING: No data loaded. Creating dummy data for code verification.")
    dummy_labels = ['ClassA', 'ClassB', 'ClassC', 'ClassD', 'ClassE']
    df = pd.DataFrame({
        'file_path': ['dummy.jpg'] * 100,
        'label': np.random.choice(dummy_labels, 100)
    })

ERROR: Directory not found: /kaggle/input/multi-cancer/Multi Cancer/Multi Cancer/Cervical Cancer
